# Econ 3916 Midterm Project — Econometric Foundation

**ECON 3916 · Applied Data Analytics** · Topics 1-14 · 100 points · your role: Independent analyst

This is the **starter**: the structure of the project in rubric order. Fill the empty cells and keep every heading so the grader finds each section. The brief on Canvas has the full text of every section and the rubric. The grader reruns this notebook from the top, so every number, table and figure in your report must come from a cell here.

**🛑 This project is entirely NO AI.** Not phases one to three — all of it. No AI for the code, the analysis, the writing, or the editing. This is the one assessment in the course that establishes you can do the work unaided, and it is the one you will draw on when an interviewer asks what you built. Everything you have practised with a co-pilot, you do here alone.

## Start here

Work top to bottom. What each part of this notebook is for:

1. **Data provenance**: where the file came from. It feeds Section 1 and your README.
2. **Section 1**: load the data into a DataFrame called `df`, then run the *Section 1 self-check* cell and answer the questions under it. Do this before the **Mon 2 Nov** checkpoint.
3. **Section 2**: figures, each with a caption that states a finding.
4. **Section 3**: the OLS model, with a draft committed by **Fri 6 Nov**.
5. **Section 4**: diagnostics, only after Topic 14.
6. **Section 5**: words only.
7. **Numbers my report quotes**: last. Every number in the report must be printed there.

The brief's *Six checks before you post* list the problems that came up most often in Checkpoint 1. Read them before you settle on your controls.

## Setup

Imports only. Add any others you need here, at the top.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path

## Data provenance

Fill these in when you download the data. They feed Section 1 and your README.

In [2]:
SOURCE_URL = ("PFR attendance 2010-2019, e.g. https://www.pro-football-reference.com/years/2019/attendance.htm; "
              "nflverse games file, https://github.com/nflverse/nfldata; "
              "Census metro population, https://www2.census.gov/programs-surveys/popest/datasets/2010-2019/metro/totals/cbsa-est2019-alldata.csv")      # the page or file you downloaded from
DOWNLOAD_DATE = "2026-11-08"    # YYYY-MM-DD
FILE_NAME = "data/attendance.xlsx"        # the file as saved in your repository's data/ folder

**Who collected it, for what purpose, what is missing and why?**


The data was collected by Pro football reference. They track NFL data, and post the attendance report for teams each week. Teams count tickets sold for attendance, instead of people in stands. Also, games played internationally still count from a home game. The win % figure comes from nfl verse, which just sorted data from Pro Football Reference, which I verified myself. Metro population comes dfrom the Census Bureau, and their yearly estimate for metro areas. There are no missing values, but demand past stadium capacity cannot be tracked. If fans wanted tickets after a game is sold out, that is demand we cannot track, and is MNAR.

## Section 1: The Question and the Data (20 pts)

State the question in one sentence, and say why it matters and to whom. Describe the data as a product of measurement decisions, name the missingness mechanism (MCAR, MAR or MNAR) and defend the label. Build a summary-statistics table a reader can read without asking you a question.

In [3]:
ATT_URL = "https://raw.githubusercontent.com/Vir-Juneja/ECON3916-MIDTERM/main/data/attendance.xlsx"
GAMES_URL = "https://raw.githubusercontent.com/nflverse/nfldata/master/data/games.csv"
CENSUS_URL = "https://www2.census.gov/programs-surveys/popest/datasets/2010-2019/metro/totals/cbsa-est2019-alldata.csv"

# attendance from 2010-2019 using PFR data, split into sheets per season
pieces = []
for year in range(2010, 2020):
    sheet = pd.read_excel(ATT_URL, sheet_name=str(year))
    print(f"{year} raw attendance table: {sheet.shape}")
    sheet["season"] = year
    pieces.append(sheet)
raw_att = pd.concat(pieces, ignore_index=True)

att = raw_att[raw_att["Tm"].notna()].copy()          # removes league total frm each season
att = att[["season", "Tm", "Stadium", "Home"]]
att["home_att_per_game"] = att["Home"] / 8            # divided by 8 as each team played 8 home games per year
print(f"Attendance rows: {att.shape}")

# Win %, using nflverse data on github which is sourced from PFR (double checked)
games = pd.read_csv(GAMES_URL)
print(f"Games file: {games.shape}")
reg = games[(games["game_type"] == "REG") & (games["season"].between(2009, 2019))]

home = reg[["season", "home_team", "home_score", "away_score"]].copy()
home.columns = ["season", "team", "pf", "pa"]
away = reg[["season", "away_team", "away_score", "home_score"]].copy()
away.columns = ["season", "team", "pf", "pa"]
long = pd.concat([home, away], ignore_index=True)     # one row per team per game

# win = 1, loss = 0, tie = 0.5
long["win"] = (long["pf"] > long["pa"]).astype(float) + 0.5 * (long["pf"] == long["pa"]).astype(float)

codes = pd.DataFrame([
    ["ARI", "Cardinals"], ["ATL", "Falcons"], ["BAL", "Ravens"], ["BUF", "Bills"],
    ["CAR", "Panthers"], ["CHI", "Bears"], ["CIN", "Bengals"], ["CLE", "Browns"],
    ["DAL", "Cowboys"], ["DEN", "Broncos"], ["DET", "Lions"], ["GB", "Packers"],
    ["HOU", "Texans"], ["IND", "Colts"], ["JAX", "Jaguars"], ["KC", "Chiefs"],
    ["MIA", "Dolphins"], ["MIN", "Vikings"], ["NE", "Patriots"], ["NO", "Saints"],
    ["NYG", "Giants"], ["NYJ", "Jets"], ["OAK", "Raiders"], ["PHI", "Eagles"],
    ["PIT", "Steelers"], ["SD", "Chargers"], ["LAC", "Chargers"], ["SF", "49ers"],
    ["SEA", "Seahawks"], ["STL", "Rams"], ["LA", "Rams"], ["TB", "Buccaneers"],
    ["TEN", "Titans"], ["WAS", "Redskins"],
], columns=["team", "franchise"])
long = long.merge(codes, on="team", how="left")
print(f"Game rows with no team match: {long['franchise'].isna().sum()}")

wins = long.groupby(["franchise", "season"])["win"].mean().reset_index()
wins = wins.rename(columns={"win": "win_pct"})

# last season win %, moved forward, then merged back
prev = wins.copy()
prev["season"] = prev["season"] + 1
prev = prev.rename(columns={"win_pct": "prev_win_pct"})
wins = wins.merge(prev, on=["franchise", "season"], how="left")

# Census population,
cen = pd.read_csv(CENSUS_URL, encoding="latin-1")     # file had latin 1 encoding
print(f"Census file: {cen.shape}")
cen = cen[cen["LSAD"] == "Metropolitan Statistical Area"]
pieces = []
for year in range(2010, 2020):
    piece = cen[["CBSA"]].copy()
    piece["season"] = year
    piece["metro_pop"] = cen[f"POPESTIMATE{year}"]
    pieces.append(piece)
pop = pd.concat(pieces, ignore_index=True)

# Joins
teams = pd.DataFrame([
    ["Arizona Cardinals", "Cardinals", 38060], ["Atlanta Falcons", "Falcons", 12060],
    ["Baltimore Ravens", "Ravens", 12580], ["Buffalo Bills", "Bills", 15380],
    ["Carolina Panthers", "Panthers", 16740], ["Chicago Bears", "Bears", 16980],
    ["Cincinnati Bengals", "Bengals", 17140], ["Cleveland Browns", "Browns", 17460],
    ["Dallas Cowboys", "Cowboys", 19100], ["Denver Broncos", "Broncos", 19740],
    ["Detroit Lions", "Lions", 19820], ["Green Bay Packers", "Packers", 24580],
    ["Houston Texans", "Texans", 26420], ["Indianapolis Colts", "Colts", 26900],
    ["Jacksonville Jaguars", "Jaguars", 27260], ["Kansas City Chiefs", "Chiefs", 28140],
    ["Miami Dolphins", "Dolphins", 33100], ["Minnesota Vikings", "Vikings", 33460],
    ["New England Patriots", "Patriots", 14460], ["New Orleans Saints", "Saints", 35380],
    ["New York Giants", "Giants", 35620], ["New York Jets", "Jets", 35620],
    ["Oakland Raiders", "Raiders", 41860], ["Philadelphia Eagles", "Eagles", 37980],
    ["Pittsburgh Steelers", "Steelers", 38300], ["San Diego Chargers", "Chargers", 41740],
    ["Los Angeles Chargers", "Chargers", 31080], ["San Francisco 49ers", "49ers", 41860],
    ["Seattle Seahawks", "Seahawks", 42660], ["St. Louis Rams", "Rams", 41180],
    ["Los Angeles Rams", "Rams", 31080], ["Tampa Bay Buccaneers", "Buccaneers", 45300],
    ["Tennessee Titans", "Titans", 34980], ["Washington Redskins", "Redskins", 47900],
], columns=["Tm", "franchise", "CBSA"])

df = att.merge(teams, on="Tm", how="left")
print(f"\nAttendance rows with no team match: {df['CBSA'].isna().sum()}")
df = df.merge(wins, on=["franchise", "season"], how="left")
df = df.merge(pop, on=["CBSA", "season"], how="left")
df["log_metro_pop"] = np.log(df["metro_pop"])

print(f"Final dataset (rows, columns): {df.shape}")
print(f"\nMissing values per column:\n{df.isna().sum()}")

2010 raw attendance table: (33, 22)
2011 raw attendance table: (33, 22)
2012 raw attendance table: (33, 22)
2013 raw attendance table: (33, 22)
2014 raw attendance table: (33, 22)
2015 raw attendance table: (33, 22)
2016 raw attendance table: (33, 22)
2017 raw attendance table: (33, 22)
2018 raw attendance table: (33, 22)
2019 raw attendance table: (33, 22)
Attendance rows: (320, 5)
Games file: (7548, 46)
Game rows with no team match: 0
Census file: (2797, 97)

Attendance rows with no team match: 0
Final dataset (rows, columns): (320, 11)

Missing values per column:
season               0
Tm                   0
Stadium              0
Home                 0
home_att_per_game    0
franchise            0
CBSA                 0
win_pct              0
prev_win_pct         0
metro_pop            0
log_metro_pop        0
dtype: int64


In [4]:
# Section 1 self-check: run once your data is loaded into a DataFrame called df.
# It prints what Section 1 has to report. It doesn't judge your choices: answer the questions below.
UNIT = "team-season"        # one row is one what? e.g. "county" or "country-year"
OUTCOME = "home_att_per_game"     # your dependent variable's column name
MAIN_X = "win_pct"      # your main explanatory variable's column name
CONTROLS = ["prev_win_pct", "log_metro_pop", "Tm", "season"]    # every other column in your model

if "df" in globals():
    print(f"Rows: {len(df):,}   Columns: {df.shape[1]}   One row is: {UNIT or '(fill in UNIT)'}")
    cols = [c for c in [OUTCOME, MAIN_X, *CONTROLS] if c]
    absent = [c for c in cols if c not in df.columns]
    if absent:
        print("Not in df:", absent)
    present = [c for c in cols if c in df.columns]
    if present:
        print("\nMissing values per column:")
        print(df[present].isna().sum().to_string())
    if OUTCOME in df.columns and pd.api.types.is_numeric_dtype(df[OUTCOME]):
        print(f"\nSkew of {OUTCOME}: {df[OUTCOME].skew():.2f} (above about 1, consider a log)")
else:
    print("Load your data into a DataFrame called df first, then run this cell.")

Rows: 320   Columns: 11   One row is: team-season

Missing values per column:
home_att_per_game    0
win_pct              0
prev_win_pct         0
log_metro_pop        0
Tm                   0
season               0

Skew of home_att_per_game: -0.49 (above about 1, consider a log)


**Answer before you go on.**

1. For each control: does your main explanatory variable change it? If yes, it is on the path from X to Y. Drop it, or run with and without and say what each version measures.
2. Are all your variables in the same units and for the same years?
3. If your question is about a policy, where is the period before it?

*Your answers here.*
1. No, the seasons win % does not change any of my controls. The last season win %, metro population, the team, and the year are all fixed and will not change with this.
2. Yes, every row is a teams season from 2010 to 2019. Attendance and population are always tracked in the same unit, people. The data starts from 2009 so we have the win % from the last season for 2010.
3. N/A, not a policy question.

In [5]:
rows = []
for col, label, scale in [
    ("home_att_per_game", "Home attendance per game (thousands)", 1000),
    ("win_pct", "Win % this season (0 to 1)", 1),
    ("prev_win_pct", "Win % last season (0 to 1)", 1),
    ("metro_pop", "Metro population (millions)", 1000000),
]:
    values = df[col] / scale
    rows.append([label, values.count(), values.mean(), values.std(), values.min(), values.max()])

summary = pd.DataFrame(rows, columns=["Variable", "N", "Mean", "Std. dev.", "Min", "Max"]).round(2)
summary

,Variable,N,Mean,Std. dev.,Min,Max
0,Home attendance per game (thousands),320,67.81,8.70,25.34,92.72
1,Win % this season (0 to 1),320,0.50,0.19,0.00,0.94
2,Win % last season (0 to 1),320,0.50,0.19,0.00,0.94
3,Metro population (millions),320,4.87,4.44,0.31,19.33


Question:
For NFL teams from 2010 to 2019, how does a teams winning percentage relate to how many fans attend home games, controlled for last seasons win percentage and the size of its metro area.

Why Does it matter. Teams use this when they set ticket prices, and when spending money in free agency. The cities care too, as stadiums are often tax-payer funded.

The summary table: The data has 320 teams seasons (32 teams * 10 years). The average home game drew 67,800 fans, roughly. The range runs from 25,300 to 92,700, the lower end being due to a chargers season with reduced stadium capacity. The win % ranges from 0, to .94 (15-1 packers, and 0-16 browns). It averages out to .500, as the amount of games played is even (if one team loses, another has to win, and ties are .500 for both). Metro population ranges from .3 million in Wisconsin to nearly 20 million in New York City and New Jersey areas. Since attendance has a slight left skew, I decided not to take its log.

## Section 2: Descriptive Evidence (20 pts)

Show the reader the data before any model. At least two figures, each with a caption that states its finding, not its variables. Chapter 3's standards for axes. If the outcome is skewed, show it and say what you will do about it.

*Your words here.*

## Section 3: The Model (25 pts)

Fit a multivariate OLS model and report it in a formatted regression table (coefficients, standard errors, N, R-squared), not a `summary()` dump. Interpret at least two coefficients in full: magnitude, units, direction, precision. For every control, say whether it blocks a confounding path or adds precision, and what would make it a bad control (Chapter 10). OLS only: no difference-in-differences estimate.

*Your words here.*

## Section 4: Diagnostics (20 pts)

Residuals against fitted values and against your main predictor; a constant-variance check, with robust standard errors if it fails; variance inflation factors, interpreted; at least one influential-observation check.

**Start only after Topic 14.** Write Sections 1 to 3 first; they do not depend on it.

*Your words here.*

## Section 5: Threats and Conclusion (10 pts)

First the threats: what would have to be true for your main coefficient to be causal, is it, and if not, what your estimate is instead. Then the conclusion: what you found, how confident a reader should be, and what you would do with another month. Half a page. No new results, so there is no code cell here.

*Your words here.*

## Appendix

The regression table, the diagnostics output and any extra figures. The appendix does not count toward the 6–8 page limit of the report.

## Numbers my report quotes

Every number in your report, each with a name that says where it comes from. A number in the report that this cell does not print is unsupported, and is graded as if it were missing. Keep it the last code cell.

In [6]:
REPORT_NUMBERS = {
    # "Section 1: rows in the final sample": ...,
    # "Section 3: coefficient on <your main predictor>": ...,
}

for name, value in REPORT_NUMBERS.items():
    print(f"{name}: {value}")

## Checkpoints

In Colab, *File › Save a copy in GitHub* makes a commit each time you use it. Save at each checkpoint so the commit history shows it.

- [ ] **Mon 26 Oct:** your question and data source, posted in the Midterm checkpoint 1 discussion.
- [ ] **Mon 2 Nov:** your data loaded and your summary table, committed to your repository.
- [ ] **Fri 6 Nov:** a draft of your model, committed.
- [ ] **Mon 9 Nov, 11:59pm:** the project is due.

## Before you submit

- [ ] A Colab notebook with all your analysis, cleanly organised.
- [ ] A written report, 6 to 8 pages of text, uploaded as a PDF.
- [ ] A GitHub repository containing both, with a `README.md` you wrote yourself.
- [ ] **Question and data provenance (20):** A defensible measurable question; data described as a measurement product; missingness mechanism identified and defended; interpretable summary table
- [ ] **Descriptive evidence (20):** Two or more figures each making a stated point; captions state findings not variables; Chapter 3 standards met
- [ ] **Model and interpretation (25):** Properly formatted regression table; two coefficients interpreted in full including units and magnitude; every control justified by role
- [ ] **Diagnostics (20):** Residuals, heteroskedasticity, VIF and influence all run and interpreted; robust errors used where warranted
- [ ] **Threats and conclusion (10):** The causal claim assessed in plain terms; the estimand correctly labelled if not causal; conclusion adds no new results
- [ ] **Repository, checkpoints and reproducibility (5):** Public GitHub repo containing the notebook and the report; the two committed checkpoints visible in the commit history; data included or its download documented; a self-written README giving the question, the source, how to run it and the headline finding
- [ ] The Colab notebook is shared with *Anyone with the link*, and the link is in the Canvas submission box.
- [ ] Every number in the report is printed by the *Numbers my report quotes* cell.
- [ ] No AI anywhere: code, analysis, writing, editing, README.
- [ ] *File › Save a copy in GitHub*, path `Midterm Project/Econ_3916_Midterm.ipynb`.
- [ ] Run *Runtime › Restart and run all* once, top to bottom, before you hand in.